# Zadanie 2B — PAMAP2

**Cel:** przygotuj wielokanałowe sygnały z czujników ubieralnych, zbuduj model rozpoznający aktywność i sprawdź, jak rozwiązanie generalizuje się na osoby niewidziane podczas treningu.

In [ ]:
from pathlib import Path
import pathlib
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from huggingface_hub import hf_hub_download
from scipy import signal
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [3]:
DATA_DIR = Path("data/zadanie-2b")
DATA_DIR.mkdir(parents=True, exist_ok=True)

archive_path = hf_hub_download(
    repo_id="AI-MED-AGH/Recruitment-Task-2B",
    filename="original/pamap2+physical+activity+monitoring.zip",
    repo_type="dataset",
    local_dir=DATA_DIR,
)
with ZipFile(archive_path) as archive:
    archive.extractall(DATA_DIR / "pamap2")

print(f"Dane zapisano w: {DATA_DIR / 'pamap2'}")

Dane zapisano w: data/zadanie-2b/pamap2


## A — Poznanie i przygotowanie danych

Rozpoznaj uczestników, aktywności, czujniki, częstotliwości próbkowania i braki danych. Uzasadnij wybór sygnałów, obsługę braków oraz sposób tworzenia okien. Podziel dane według uczestników i dopilnuj, aby żadna osoba nie występowała w więcej niż jednym podziale.

In [4]:
dataset_path = Path.cwd() / "data" / "zadanie-2b" / "pamap2"
dataset_zip = dataset_path / "PAMAP2_Dataset.zip"
with ZipFile(dataset_zip, "r") as f:
    f.extractall(dataset_path)

In [5]:
dataset_path = Path.cwd() / "data" / "zadanie-2b" / "pamap2" / "PAMAP2_Dataset" / "Protocol"
all_subjects = list(dataset_path.iterdir())
train_subjects = all_subjects[:5]
test_subjects = all_subjects[5:7]
val_subjects = all_subjects[7:9]
for file in all_subjects[:1]:
    df_test = pd.read_csv(file, delimiter=" ")
df_test.iloc[:,1].value_counts()

0
0     126459
1      27187
6      23575
17     23573
2      23480
16     22941
4      22253
3      21717
5      21265
7      20265
12     15890
13     14899
24     12912
Name: count, dtype: int64

In [6]:
#informacja o czasie jest bezwartościowa w kontekście klasyfikacji aktywności, więc zostanie odrzucona.
#W readme napisano zostało również że drugi akcelerometr jest źle skalibrowany więc również zostanie odrzucony
#z każdego IMU sensora. Uznałem że rekordy w których jest więcej niż 1/3 atrybutów pusta, będą odrzucane,
#a reszta wartości nan będzie interpolowana liniowo.
activities = list(range(1, 14)) + list(range(16, 21)) + [24]
def preprocess_subject(file: Path):
    df = pd.read_csv(file, header=None, sep=r"\s+")
    df = df[df.iloc[:, 1].isin(activities)].copy()
    correct_indicies = [1, 2]
    for i in [3, 20, 37]:
        correct_indicies.extend(list(range(i, i + 4)) + list(range(i + 7, i + 17)))
    df = df.iloc[:, correct_indicies].copy()
    X = df.iloc[:, 1:].to_numpy()
    y = df.iloc[:, 0].to_numpy()
    return X, y

def prepare_windows(files: list[Path], n: int = 100, step: int = 50):
    X_windows = []
    y_windows = []
    for file in files:
        X, y = preprocess_subject(file)
        data = pd.DataFrame(np.concatenate([y[:, np.newaxis], X], axis=1))
        data["block"] = (data.iloc[:,0] != data.iloc[:,0].shift()).cumsum()
        for _, block in data.groupby("block"):
            for i in np.arange(0, block.shape[0] - n + 1, step):
                X_window_activity = block.iloc[i:i + n, 1:]
                y_window_activity = block.iloc[i, 0]
                if (X_window_activity.isna().sum(axis=1) > X_window_activity.shape[1] / 2).sum():
                    continue
                X_window_activity = X_window_activity.interpolate("linear", axis=0).ffill().bfill().fillna(0).copy()
                X_windows.append(X_window_activity.to_numpy())
                y_windows.append(y_window_activity)
    return np.array(X_windows), np.array(y_windows)

In [36]:
# X, y = prepare_data(test_subjects)
# print(pd.DataFrame(y).value_counts())

X_test, y_test = prepare_windows(test_subjects)
X_train, y_train = prepare_windows(train_subjects)
X_val, y_val = prepare_windows(test_subjects)

In [37]:
activity_mapper = {activities[i] : i for i in range(len(activities))}
y_test = np.array([activity_mapper[x.item()] for x in y_test])
y_val = np.array([activity_mapper[x.item()] for x in y_val])
y_train = np.array([activity_mapper[x.item()] for x in y_train])

In [39]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[2])).reshape(X_train.shape)
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[2])).reshape(X_test.shape)
X_val = scaler.transform(X_val.reshape(-1, X_val.shape[2])).reshape(X_val.shape)

In [40]:
class ActivitiesDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        super().__init__()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, index: int):
        return self.X[index], self.y[index]

In [41]:
train_loader = DataLoader(ActivitiesDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(ActivitiesDataset(X_test, y_test), batch_size=32)
val_loader = DataLoader(ActivitiesDataset(X_val, y_val), batch_size=32)

## B — Modelowanie i walidacja

Wytrenuj model rozpoznawania aktywności. Przeanalizuj cechy lub architekturę, procedurę treningową i walidację z rozłącznymi uczestnikami. Wszystkie transformacje dopasowuj wyłącznie na danych treningowych, a niewidzianych uczestników testowych zachowaj do końcowej oceny.

In [42]:
class PamapModel(nn.Module):
    def __init__(self, in_channels: int, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 64, 3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2, 2)
        )
        self.gru = nn.GRU(input_size=64, hidden_size=128, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        out, _ = self.gru(x)
        out = out[:,-1,:]
        return self.classifier(out)


In [43]:
from copy import deepcopy
model = PamapModel(X_train.shape[2], len(activities))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002)
epochs = 5
best_model = deepcopy(model.state_dict())
best_loss = float("inf")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for X, y in train_loader:
        X = X.type(torch.float32)
        preds = model(X)
        y = y.squeeze().type(torch.long)
        loss = loss_fn(preds, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.inference_mode():
        for X, y in val_loader:
            X = X.type(torch.float32)
            preds = model(X)
            y = y.squeeze().type(torch.long)
            loss = loss_fn(preds, y)
            val_loss += loss.item()
            _, outputs = torch.max(preds, dim=1)
            correct += (y == outputs).sum()
            total += X.shape[0]

    val_loss = val_loss / len(val_loader)
    acc = correct / total
    if val_loss < best_loss:
        best_loss = val_loss
        best_model = deepcopy(model.state_dict())
    if val_loss < 0.01:
        break

    print(f"epoch: {epoch + 1}, val_loss: {val_loss}, acc: {acc}")
model.load_state_dict(best_model)

epoch: 1, val_loss: 0.40999506814319375, acc: 0.8827549815177917
epoch: 2, val_loss: 0.31705851347322206, acc: 0.918612003326416
epoch: 3, val_loss: 0.25054807136867696, acc: 0.9315457344055176
epoch: 4, val_loss: 0.3441555289068338, acc: 0.917350172996521
epoch: 5, val_loss: 0.3716192245595956, acc: 0.8943217396736145


<All keys matched successfully>

## C — Ocena i wyjaśnialność

Oceń model na niewidzianych uczestnikach, używając odpowiednich metryk wieloklasowych oraz analizy wyników dla klas i osób. Znajdź i zacytuj metodę wyjaśniania modeli sygnałowych lub szeregów czasowych, zastosuj ją do poprawnych i błędnych predykcji kilku aktywności i omów ograniczenia.

In [56]:
model.eval()
preds = []
labels = []
with torch.inference_mode():
    for X, y in test_loader:
        y_preds = model(X.type(torch.float32))
        _, output = torch.max(y_preds, dim=1)
        preds.extend(output.tolist())
        labels.extend(y.squeeze().tolist())

(np.array(preds) == np.array(labels)).sum() / len(preds)

np.float64(0.9315457413249212)

## Oddanie zadania

Zapisz kod, wyniki, wykresy i uzasadnienia w tym notatniku. Podaj użyte źródła oraz narzędzia AI i zadbaj o odtwarzalność rozwiązania. Umieść rozwiązanie na swoim repozytorium.